In [ ]:
DATASET_NAME = "mimic_iv_cohort"

In [22]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '..'
  SRC = f"{PROJECT_ROOT}/src"
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
  if SRC not in sys.path:
    sys.path.append(SRC)

In [23]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from src.config import Config

LOCAL_MIMIC_DB = f"{PROJECT_ROOT}/{Config.DATA_DIR}/mimic_local.db"
conn = duckdb.connect(database=LOCAL_MIMIC_DB)

In [24]:
def load_query(file_path: str) -> str:
  with open(file_path, 'r') as f:
    return f.read()

# Exploration

In [25]:
MIMIC_PATH = f"{PROJECT_ROOT}/{Config.DATA_DIR}/mimic_iv/physionet.org/files/mimiciv/3.1"
MIMIC_ICU_PATH = f"{MIMIC_PATH}/icu"
MIMIC_HOSP_PATH = f"{MIMIC_PATH}/hosp"

In [26]:
d_items_df = pd.read_csv(f'{MIMIC_ICU_PATH}/d_items.csv.gz')
# item_filter = (d_items_df['label'].str.lower().str.contains("asopressin"))
item_filter = (d_items_df['itemid'] == 223761)
# item_filter = (d_items_df['linksto'] == "outputevents")
# item_filter = (d_items_df['category'] == "Dialysis")
d_items_df[item_filter]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
337,223761,Temperature Fahrenheit,Temperature F,chartevents,Routine Vital Signs,°F,Numeric,NaN,NaN


In [27]:
labitems_df = pd.read_csv(f'{MIMIC_HOSP_PATH}/d_labitems.csv.gz')
lab_filter = (labitems_df['itemid'] == 50885)
labitems_df[lab_filter]

,itemid,label,fluid,category
83,50885,"Bilirubin, Total",Blood,Chemistry


# Cohort extraction

## Exclusions and target definition

In [28]:
cohort_query = load_query("mimic_queries/select_cohort.sql")

final_cohort = conn.execute(cohort_query).df()

conn.register("cohort", final_cohort)

print(f"Final Cohort: {final_cohort.shape[0]}")
print(f"Ethnic proportions: {final_cohort["race"].value_counts(normalize=True)}")
print(f"Case cohort: {final_cohort['target_noaf'].sum()}")
print(f"Control cohort: {final_cohort.shape[0] - final_cohort['target_noaf'].sum()}")

Final Cohort: 19480
Ethnic proportions: race
WHITE                                        0.574076
UNKNOWN                                      0.135883
BLACK/AFRICAN AMERICAN                       0.083676
OTHER                                        0.037988
UNABLE TO OBTAIN                             0.030082
WHITE - OTHER EUROPEAN                       0.019713
ASIAN                                        0.013296
HISPANIC/LATINO - PUERTO RICAN               0.012423
ASIAN - CHINESE                              0.011191
HISPANIC/LATINO - DOMINICAN                  0.009600
BLACK/CARIBBEAN ISLAND                       0.008316
HISPANIC OR LATINO                           0.008008
WHITE - RUSSIAN                              0.006571
BLACK/CAPE VERDEAN                           0.006520
PATIENT DECLINED TO ANSWER                   0.005647
BLACK/AFRICAN                                0.005493
PORTUGUESE                                   0.003953
ASIAN - SOUTH EAST ASIAN             

## Feature extraction

### Vitals

In [29]:
vitals_query = load_query("mimic_queries/extract_vitals.sql")
df_vitals = conn.execute(vitals_query).df()

### Lab measurements

In [30]:
labs_query = load_query("mimic_queries/extract_lab_measurements.sql")
df_labs = conn.execute(labs_query).df()

### Urine output

In [31]:
urine_query = load_query("mimic_queries/extract_urine_outputs.sql")
df_urine = conn.execute(urine_query).df()

### Interventions

In [32]:
interventions_query = load_query("mimic_queries/extract_interventions.sql")
df_interventions = conn.execute(interventions_query).df()

### Comorbidities

In [33]:
comorbidities_query = load_query("mimic_queries/extract_comorbidities.sql")
df_comorbidities = conn.execute(comorbidities_query).df()

# Full cohort and features

In [34]:
dataset = final_cohort.merge(df_vitals, on="stay_id", how="left")
dataset = dataset.merge(df_labs, on="stay_id", how="left")
dataset = dataset.merge(df_urine, on="stay_id", how="left")
dataset = dataset.merge(df_interventions, on="stay_id", how="left")
dataset = dataset.merge(df_comorbidities, on="hadm_id", how="left")

comorb_cols = [col for col in dataset.columns if col.startswith('comorb_')]
dataset[comorb_cols] = dataset[comorb_cols].fillna(0).astype(int)

print(dataset.describe().T.to_markdown())

|                         |   count | mean                       | min                 | 25%                        | 50%                 | 75%                        | max                 |             std |
|:------------------------|--------:|:---------------------------|:--------------------|:---------------------------|:--------------------|:---------------------------|:--------------------|----------------:|
| subject_id              |   19480 | 14998472.265913758         | 10000690.0          | 12504284.5                 | 15003625.0          | 17521308.75                | 19999840.0          |     2.89105e+06 |
| hadm_id                 |   19480 | 24971986.08834702          | 20001361.0          | 22489511.5                 | 24968827.5          | 27446870.0                 | 29999098.0          |     2.87746e+06 |
| stay_id                 |   19480 | 34944378.27736139          | 30000646.0          | 32406347.0                 | 34894210.0          | 37461555.5              

## Harmonise demographics

In [35]:
def harmonize_demographics(df):
  race_map = {
    'WHITE': 0, 'WHITE - OTHER EUROPEAN': 0, 'WHITE - RUSSIAN': 0, 
    'WHITE - EASTERN EUROPEAN': 0, 'WHITE - BRAZILIAN': 0, 'PORTUGUESE': 0,
    
    'BLACK/AFRICAN AMERICAN': 1, 'BLACK/CARIBBEAN ISLAND': 1, 
    'BLACK/CAPE VERDEAN': 1, 'BLACK/AFRICAN': 1,
    
    'ASIAN': 2, 'ASIAN - CHINESE': 2, 'ASIAN - SOUTH EAST ASIAN':2, 
    'ASIAN - ASIAN INDIAN': 2, 'ASIAN - KOREAN': 2,
    
    'HISPANIC/LATINO - PUERTO RICAN': 3, 'HISPANIC/LATINO - DOMINICAN': 3, 
    'HISPANIC OR LATINO': 3, 'HISPANIC/LATINO - GUATEMALAN': 3, 
    'HISPANIC/LATINO - SALVADORAN': 3, 'HISPANIC/LATINO - MEXICAN': 3, 
    'HISPANIC/LATINO - HONDURAN': 3, 'HISPANIC/LATINO - CENTRAL AMERICAN': 3, 
    'HISPANIC/LATINO - COLUMBIAN': 3, 'HISPANIC/LATINO - CUBAN': 3,
    'SOUTH AMERICAN': 3,
  }
  
  # Map matching text; classify sparse or undisclosed tracks as -1
  df['race_group'] = df['race'].map(race_map).fillna(-1).astype(int)
  
  # Convert sensitive categoricals to clean reference strings
  df['gender'] = df['gender'].map({'M': 0, 'F': 1}).astype(int)
  
  return df

dataset = harmonize_demographics(dataset)
print(dataset.groupby(['race_group'])["target_noaf"].value_counts())

race_group  target_noaf
-1          0               3839
            1                331
 0          0              10850
            1               1014
 1          0               1946
            1                 80
 2          0                580
            1                 44
 3          0                771
            1                 25
Name: count, dtype: int64


## SOFA

In [36]:
def calculate_sofa(df):
  sofa_df = df.copy()

  # RESPIRATORY SYSTEM
  fio2_fraction = sofa_df['fio2_max'] / 100.0
  pf_ratio = sofa_df['pao2_min'] / fio2_fraction
  sf_ratio = sofa_df['spo2_min'] / fio2_fraction
  resp_ratio = pf_ratio.fillna(sf_ratio)
  is_vented = sofa_df['vent_day1'] == 1

  resp_conds = [
    (resp_ratio <= 100) & is_vented,
    (resp_ratio <= 200) & is_vented,
    (resp_ratio <= 300),
    (resp_ratio <= 400),
    (resp_ratio > 400) | resp_ratio.isna()
  ]
  resp_choices = [4, 3, 2, 1, 0]
  sofa_df['sofa_respiratory'] = np.select(resp_conds, resp_choices, default=0)

  # COAGULATION
  coag_conds = [
    (sofa_df['platelets_min'] <= 20),
    (sofa_df['platelets_min'] <= 50),
    (sofa_df['platelets_min'] <= 100),
    (sofa_df['platelets_min'] <= 150),
    (sofa_df['platelets_min'] > 150) | sofa_df['platelets_min'].isna()
  ]
  coag_choices = [4, 3, 2, 1, 0]
  sofa_df['sofa_coagulation'] = np.select(coag_conds, coag_choices, default=0)

  # LIVER
  liver_conds = [
    (sofa_df['bilirubin_max'] >= 12.0),
    (sofa_df['bilirubin_max'] >= 6.0) & (sofa_df['bilirubin_max'] < 12.0),
    (sofa_df['bilirubin_max'] >= 2.0) & (sofa_df['bilirubin_max'] < 6.0),
    (sofa_df['bilirubin_max'] >= 1.2) & (sofa_df['bilirubin_max'] < 2.0),
    (sofa_df['bilirubin_max'] < 1.2) | sofa_df['bilirubin_max'].isna()
  ]
  liver_choices = [4, 3, 2, 1, 0]
  sofa_df['sofa_liver'] = np.select(liver_conds, liver_choices, default=0)

  # CARDIOVASCULAR SYSTEM
  cardio_conds = [
    # Score 4: High dose vasopressor rates
    (sofa_df['max_dopamine_rate'] > 15.0) | 
    (sofa_df['max_epinephrine_rate'] > 0.1) | 
    (sofa_df['max_norepinephrine_rate'] > 0.1),
    
    # Score 3: Moderate dose vasopressor rates
    (sofa_df['max_dopamine_rate'] > 5.0) | 
    ((sofa_df['max_epinephrine_rate'] > 0.0) & (sofa_df['max_epinephrine_rate'] <= 0.1)) | 
    ((sofa_df['max_norepinephrine_rate'] > 0.0) & (sofa_df['max_norepinephrine_rate'] <= 0.1)),
    
    # Score 2: Low dose dopamine or presence of dobutamine proxy
    (sofa_df['max_dopamine_rate'] > 0.0) & (sofa_df['max_dopamine_rate'] <= 5.0),
    
    # Score 1: Mean Blood Pressure hypotension under 70 mmHg (without vasopressors)
    (sofa_df['mbp_min'] < 70.0),
    
    # Score 0: Stable hemodynamics
    (sofa_df['mbp_min'] >= 70.0) | sofa_df['mbp_min'].isna()
  ]
  cardio_choices = [4, 3, 2, 1, 0]
  sofa_df['sofa_cardiovascular'] = np.select(cardio_conds, cardio_choices, default=0)

  # CENTRAL NERVOUS SYSTEM
  # Glasgow Coma Scale
  gcs_total = sofa_df['gcs_eye_min'] + sofa_df['gcs_verbal_min'] + sofa_df['gcs_motor_min']
    
  cns_conds = [
    (gcs_total < 6),
    (gcs_total >= 6) & (gcs_total <= 9),
    (gcs_total >= 10) & (gcs_total <= 12),
    (gcs_total >= 13) & (gcs_total <= 14),
    (gcs_total == 15) | gcs_total.isna()
  ]
  cns_choices = [4, 3, 2, 1, 0]
  sofa_df['sofa_cns'] = np.select(cns_conds, cns_choices, default=0)

  # RENAL SYSTEM
  # Compute Creatinine Path
  creat_conds = [
    (sofa_df['creatinine_max'] >= 5.0),
    (sofa_df['creatinine_max'] >= 3.5) & (sofa_df['creatinine_max'] < 5.0),
    (sofa_df['creatinine_max'] >= 2.0) & (sofa_df['creatinine_max'] < 3.5),
    (sofa_df['creatinine_max'] >= 1.2) & (sofa_df['creatinine_max'] < 2.0),
    (sofa_df['creatinine_max'] < 1.2) | sofa_df['creatinine_max'].isna()
  ]
  creat_choices = [4, 3, 2, 1, 0]
  score_creat = np.select(creat_conds, creat_choices, default=0)
  
  # Compute Urine Output Path
  urine_conds = [
    (sofa_df['urine_output_total'] < 200.0),
    (sofa_df['urine_output_total'] < 500.0),
    (sofa_df['urine_output_total'] >= 500.0) | sofa_df['urine_output_total'].isna()
  ]
  urine_choices = [4, 3, 0]
  score_urine = np.select(urine_conds, urine_choices, default=0)

  sofa_df['sofa_renal'] = np.maximum(score_creat, score_urine)

  # TOTAL SCORE
  sofa_df['sofa_total'] = (
    sofa_df['sofa_respiratory'] + 
    sofa_df['sofa_coagulation'] + 
    sofa_df['sofa_liver'] + 
    sofa_df['sofa_cardiovascular'] + 
    sofa_df['sofa_cns'] + 
    sofa_df['sofa_renal']
  )

  return sofa_df

dataset = calculate_sofa(dataset)
print(dataset[['sofa_total', 'sofa_respiratory', 'sofa_cardiovascular']].describe())

         sofa_total  sofa_respiratory  sofa_cardiovascular
count  19480.000000      19480.000000         19480.000000
mean       6.595431          1.530852             1.443429
std        4.583772          1.575250             1.408569
min        0.000000          0.000000             0.000000
25%        3.000000          0.000000             1.000000
50%        6.000000          2.000000             1.000000
75%       10.000000          3.000000             1.000000
max       24.000000          4.000000             4.000000


# Save dataset

In [37]:
# Filter race groups to keep viable subgroups
cohort_filtered = dataset[dataset['race_group'].isin([0, 1])].copy()

# Target and sensitive attribute
target_col = 'target_noaf'
sens_col = 'race_group'

# Metadata columns
metadata_cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'los', 'admittime', 'race']

features = [col for col in cohort_filtered.columns
  if col not in metadata_cols + [sens_col, target_col]]

print(f"Total Cohort Size (White + Black): {cohort_filtered.shape[0]}")
print(f"Number of clinical predictive features extracted: {len(features)}")


Total Cohort Size (White + Black): 13890
Number of clinical predictive features extracted: 69


In [39]:

cohort_filtered[features + [target_col, sens_col]].to_csv(f"{PROJECT_ROOT}/{Config.DATA_DIR}/{DATASET_NAME}.csv", index=False)